# JerryscanAI PatchCore training

This notebook mirrors `training/models/train_patchcore.py` for Kaggle/Colab. The script is the canonical local entry point because it verifies every sample ID against `split_v2.csv` and records experiment metadata. This notebook uses the explicit validation folder and never loads the locked test folder during training.

In [ ]:
!pip install -q anomalib==2.2.0 lightning==2.6.1 openvino opencv-python-headless pillow

In [ ]:
from pathlib import Path

import torch
from anomalib.data import Folder
from anomalib.data.utils.split import TestSplitMode, ValSplitMode
from anomalib.engine import Engine
from anomalib.models import Patchcore
from lightning import seed_everything

In [ ]:
# Change this to the materialized split_v2 dataset location.
dataset_root = Path("G01_split_v2")
train_dir = dataset_root / "train" / "normal"
val_dir = dataset_root / "val" / "normal"
locked_test_dir = dataset_root / "test" / "normal"

angle = "G01"
model_set = "PatchcoreRaw256"
image_size = 256
batch_size = 32
num_workers = 2
backbone = "wide_resnet50_2"
layers = ("layer2", "layer3")
coreset_sampling_ratio = 0.1
num_neighbors = 9
seed = 42
results_dir = Path("working/results")
models_dir = Path("working/models")

extensions = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
def image_count(folder):
    return sum(1 for path in folder.rglob("*") if path.suffix.lower() in extensions)

counts = {"train": image_count(train_dir), "val": image_count(val_dir), "test": image_count(locked_test_dir)}
assert counts == {"train": 1997, "val": 994, "test": 1000}, counts
print(counts)

In [ ]:
seed_everything(seed, workers=True)
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
print("Accelerator:", accelerator)

pre_processor = Patchcore.configure_pre_processor(image_size=(image_size, image_size))
model = Patchcore(
    backbone=backbone,
    layers=layers,
    pre_trained=True,
    coreset_sampling_ratio=coreset_sampling_ratio,
    num_neighbors=num_neighbors,
    pre_processor=pre_processor,
)

# Folder has no val_dir argument in Anomalib 2.2. Mirror the explicit
# validation folder for fit-time validation. The real test folder is unused.
datamodule = Folder(
    name=angle,
    normal_dir=train_dir,
    normal_test_dir=val_dir,
    train_batch_size=batch_size,
    eval_batch_size=batch_size,
    num_workers=num_workers,
    test_split_mode=TestSplitMode.FROM_DIR,
    val_split_mode=ValSplitMode.SAME_AS_TEST,
    seed=seed,
)

output_dir = models_dir / model_set
output_dir.mkdir(parents=True, exist_ok=True)
output_ckpt = output_dir / f"{angle}.ckpt"
engine = Engine(accelerator=accelerator, devices=1, default_root_dir=results_dir, logger=False)
engine.fit(model=model, datamodule=datamodule)
engine.trainer.save_checkpoint(output_ckpt)
print("Saved checkpoint:", output_ckpt)